# 05 · Evaluation & Visualization

**Introduction to Pathology MIL — Notebook 5 of 5**

Measure performance *honestly* and *see* what the model learned:
- metrics table (**AUROC, AUPRC, balanced accuracy**) with **mean ± std** over folds,
- **ROC curve** and **confusion matrix**,
- **UMAP** of patch embeddings (PCA fallback if `umap-learn` isn't installed),
- recap of the attention-overlay sanity check.

## ⚙️ Colab setup

Run the cell below **first**. It writes the helper modules (`mil_models.py`, `mil_utils.py`) and regenerates any prerequisite data, so this notebook runs **standalone** in Google Colab — just choose *Runtime → Run all*. Colab already includes torch, numpy and matplotlib.

In [ ]:
# === Colab setup — RUN THIS CELL FIRST ===
# Makes the notebook self-contained: writes the helper modules used below.
# Colab already ships torch / numpy / matplotlib, so no pip install is needed.
import os, base64, gzip

_FILES = {
    'mil_models.py': 'H4sIADC+ImoC/91Z3W7byBW+11McKBdLeiXGEpBiYdQF4m22WcDxLmInN4aXGIkjcSpyRp0ZWnbSAL0qFnu5KFD0efomeZJ+M0NSpKx0nR8DRXlhUvNz5vx+55zxcDgclKJIS5XxwiTrW3r/t3/QS77gmss5pxffnxJbLjVfMqu0oYXSZHNOPzKbq0Itb/2Kuaq04clgcJFzw4lpjqFyzeZ2RJqzjM0KPqKDudKaz+0BiXJd8JJLy6xQ0lDJmbSBNmfzXMjliKSyA0aZVuuxkKCyLtjc72l5UFoshWQFfXv69IVboRJ6ds31LXlpqGRrQ4xmbElqMVgzO89pwZmtNDdEz+n9zz/Ty5/env37n9k7IqvIFCLjBKmENSNiMqNok0MRxNbrQsydFDHOAQFpBo6DNdfjQJdZC9YgDF1DQjBYGZ55TnOc6DiBdghPS91aOSJ+YzWj48Bw9Dym+nlEz48oOhtRhqH3f/+12YfBCVSTzgtmDDcxNM6NxUk42NxK8GTF3ElsCEqrrIBNS7biaTuZZswyw20yGML0C61KStNF5ZSSps4wSluIDu0H2wzqIcg0z3s/EimJGZJydzRZVHLuNsM0WPDdYDB4ROMv94DaYUInkKIQkh9576HHMPcNrZXC2JIokooKzrR0RmuH10yzkluuobkvy5I3CP2IcxAPEXTwQmUV3OXIGx2qfgEmx3BZx+bY8eNdGJYauQ9okpwwTJOnJBaC64Qu/JIgp3PQGXwpcXbzVDO+gO2EFDZNIyxajGD0NBPlEd624yb+N7xsOvKOdkTGavwcOsUNR5SLLOOyWfRkMq25do+p4ONRnLTnxO2UIw2rO4rO2d425IaQcPhuSwGMJX5RcPP+xFqrP2MCCjvnf6lcCLHCqe/UKyMK8jQsxiO38CU/fRWFzz8CHVRlo8Nk+iSO+5S3egz0a4qBUj+GGmUiXDdMZ7UuEYHBoy+4NEp3lJKDYst9J2p3nkc+gmvW281vsDlHUN5EkOz4ME6uWVEBkMSiq6rjoEYCKHO3HKqt13u6tRRbqgEeGr62skdvkkoaaJa/4dFh/CFWA7eAlm+3JB9RJQVUUtKwRbchGUWZ2kh4EGdlA25AezBtADcFbZRema2TAObAVVDkoiqKKHqemJyt+eXh1QhWnCSHiN3OGCxxLeb8+HkSPrYMBeTdQdC37x4AXiYJPT1xic0lw1b4sQvFzCe86HtnFw6kLJIRTQ8n30AUpEhMXzMtkM0eBF/+5E542vCzD2ZYKpwTqAXMcpOKiDY/XdAlWSbz6DXlqYjp/S//IiOWpRJZ9CoMXVF8f1jpw8X0ye9GPksjDo9oUSjmRn1A3gtFvL++/kgIuHDi9CGgZmEXBV59JOXzWjH3Ir7ZiyyTDyJKvosovfDLQ8YPXHVgtgnqTRR0FeUxHdTS4TvewZtJjLiVCFtWiDfwR4Oqi5seucBE7SQRg/1qZNkDXyBnqtIlnwkpFFdN2tqNSoYwDE7qI2efb3rv7VRKTVb+ek/q+6wkdyehBbBI73juR7jowyWqGiJ3Yrvxpg7j/3MJ7oM5rvVZx73z19/YP+mnx4jBwfM4geOFpDeiFedr93mhKx53E1Z9dAelN1wsc+dp2H6P/Bj/Vh7cTTtJk04n8QOln2niW5rx+YlPQMuduPkaDmyA6HM+nhcVyn/toohVN6IQDN1PoYwBFTzRadUmqTPf+NCJUCXP0MwU9EyifeJh9/RwOnmYmthJkp6f7AME/z7H8QUfzzQEyv1qJN8sQ+cm6aARtAMNByFUr4XLxbNbj0cWTeLKNWye4kxZq0o34FSWYZnv0VBjRWvDq0yNCzbjReEmlBFWXHOUINK1uPiMQ9kNzKwKC948yZYPp1vMLasCaR7B5btE1C56jRjyvGzthMIooTNlUXBb30YKQ2c/eHqrsavoDK3RtWo155ANRnDGdiSGW7MO3Sab48/Wvi0zaORWSavMLwiXgztxuh8/EZepYa6Zbwa/+QhIbfZiW/PZX9Ay6wCu+f7/geUt4ii0dzMhe8bt0IEaw0+CQI/BMKlFGNg5u9mcbjebwEUIvVNhbHR5l6lp7K8qUtfHIQ6XPNqyedXJH2l7gIuDpqqBNkaBnVRkNx0HgFdehMhsAtFdZdQx2ITeaBuvndkmGpMGJtxz5lunplNoh1eusxQy6jkVAI8eP6ap66vO6A+oXnwr1ck0mFjR72ly1PP2Gu3zRPJN+oZrZaIo7mc2llwLvonGHVoAoLauwvfKFVWrGPYA0nZcFpL2l433r4Ma23VzBpNhsdfTVedIppfcp7TeOvcJdzLRCmWdvV3z4zBWKHepVvdWedNb7Yn07hO2Bi3ci97VhzrSfZ552frMVZRfulfcZODpyjnlbgr+LpmDE5MiFhG4t1GTkmtV3LfQGZHH/wBZfwVCSwdB7vVJJdBn10CfUwT1q6BPKHf29f37yguf+Jqb18iCgxFt3MWV1QzwLpdxDVEhkW9dGbQ8WvjY3R9UiERvEZfrJELEmaIflV0qXqQdJAoYBHtGnlB83+pt2CMzPNoe5Oo650mzShRZuB6PJCvD5dk9kurBwWpTu5Pb5lAYL4TNxuXEQS22n+pfnuHtGqPhu6NdGZrbxSaptceGa73jQGMfbXZT38t9MmVQ6BN2N1RsVopieIdaaAD30PI6ucse5kvHn3unZraHv6aG/K804YcA+NfuNu2Z1kpHi2ElV1JtZP3/gK/eujPffQVJ/gPZBRiT9xgAAA==',
    'mil_utils.py': 'H4sIADC+ImoC/71a247bxhm+11MMFKQh1xItrbtBqoQBNm4SGIidRbxuUggCM0uOpLEokpmhdlc2DOS21y3QJ+iT9E3yJP3+OfAg7dpNYVcXXmk4/+mb/zj0cDgcbGWe7GqZ66jas99+/Qd7vuZKZGwt8koozZalYvVasAter8u8XO3Z0yffsbTcKS1YUdbiqiw3OhoMLlR5LTOhZwPGTtiWb0Si9wVIa5kmGa+5FjUzEpTgudRYZs0GthS83inBTq74Sp8wXTJxLdS+kQCmjKldoZkosnFdjvGH3UjotKtZVt4UeckzWazY5eNvz9lDxtm3Fy8i9lRuZaqNAcGzf/8zC1ktCl3CLm45QhWYuCugoCwLME/LTCh2U+7yjFWqzHapGBlBYFnlvKiBzbDebUs1ZFquCp5HxuAKDERRJ7pW+LaUIks2yxJMyOJc8A1fibHmS8Ee/4XpKpd1DTmB+cau9p4+tNzARBZJWQiYIq55vuO1IH0Nt60s5BZ600GYjW4Ty8uy0pZBWm6rXS2SragVIdB8iMH5ix++fzxiVxz2pLCHp+lO8XQ/wpOLH/BE1ClO9BKotSe0EoWAaXCHXNS6PXo2OWUchzE583ixE4LVGQH2W0Ca6wFtck+sVgxSr4WO2JMCpDyFFMH25c6hrwTwxsrPd/vSzwNzKuQvTNxWpaKTuWo9hk2mbKnKLayrdwDnx+dPgM0QLm9Wk2S5I49LEia3RA0jQGrcQA/cUrHbIio4rK0GH7Fn319+PWNAIF0zqR0ZhOb8lcz3TBYa/m98zVgOb3xojmW5K1LDlty6XvOa9oAfGTKuYKbBz5+UD7ybUm3AE25nJI6XSgj457VUZbGFp+CAwGP8/j7g9rw57aEPSMJ3aLW+Emt+LVguN+IwcMY+cBCO8Lv3rFgmlvcklKBIXODo+LPJCHglmdzGZ9PTEdNCZPFkNGD3fxBJRJ+uhY5PJ6Df8ttm4U+TtxNXpU6Wxm3LIp5EZyOXD+Jp9Gk4M5TkbfT3BwEscf6cUeJj5ZJlMq31iFGE47SZzuE6s0baa59NZDayz8y3nF+JfORzpUZKG2VsicxXPzoNR4j5UmVm9TR8MzDMztkJ1JS1vBYnlhF2FTW8k5TRyCI4RWcDqeUzm8OAZYrfFC6MDEO9lkvy+W/5TmvJCxaQv/vEaAEIP2eFWHESamVqLxTBWYiIXa4pfLRhKG4hHdFDbCihca0RdKTPzKzB/yjYvBVMLpfst7/9a0rhVlP66j6NeqgrlIMYoRspxFe5jeBGfJfXCdYDco7Q7PoIQCzlLbS32xwGOCElDC5DikIfDrpCSjJ0Zhe8TUEGOEakTcZVlhSlAq6BdcUw4rreVyIYunMahgfUD42KuQRwq4hog+ZZaA+RfB1C5gvziwpyRSpB3ZXoBEDY+o9xFJDIog5IN2tZELIvel4bNgRF4g7K2gI6sUISCqYj9igkjKa//fr3U3+a5LFOasOB1NJdtezejlL0edZRqhHSicJeBIZhj/ane4CGv7u4D+9Am5S/4ulmpShZsVrCv0SPr1x6vGI2nR1F/EfWu9twKdOdplixbuIjJTgDQtOzj80T67fhEa8isUQxmUngEhbPUK7Jrl0hl3T8k2iCXDKJpmdheMxCZrcOh3RdomCS9Y7tyJfM+Buea3FM+9Mc1Av2IHaBSo2G97WjzfbJluuNDaJXQpWapGWEcIwSS4He7rK8Y3apDvBF7Rez/4n9oH8OS5QAdjr+s8XcpjuETQ30S6qTKyUzFpAjrhGtW17pPgdTnq3/QWAqZE5/9S8KZ3AI9R6ueKutapm8RgtDe7l17mfhyDDrk7j0a0jgo+kmmN+Cy14vRozfSh1PW/eEDsY5T9jp2ac9LhTsEa8q9LcBFYmgLQXxcnjxuppNHmVvhm1Z6Kwmz19rPHpb0XK5IT4oJfFPvnzE9s87mLSnF7dfHYTKFDtjyKjjX++/W/mu21WncB89RrslbUcyY01rfXF++eTrZ5eArOnNadlA8AE6lfsmgcAiUiT0Q8dnvkNpe4W/SoG+N7DNP8JpxGAPfQlZee27BPgv/IJSZSHMopklaPThXnT0eyugo2PjL10qDBpen2if9TUNhia32T1roVwE+AoEWa/fNEVKIpapIAh00TQ5CANApyB4sgiNnNMsyObD1uGHiJzXQyNtOGN4ZL9idQhQsDRfvAmP2M0PeCzmZvfCR5V0SksTrNSQBU0JHTQR4gKZK8X3wbzhXS0aLdpKTKwWrlibw6VqPbcbkm5NNA/Dto6DEz2GHCT/X3YisKI7GK1URcxIwlxajtKR3NABBF7ZmJiF88li0dCaYrneLZe5CMAn7NXqlyNSu38+tKmfqY3C85fsY++2DYogbixWbHlsZcuInNihTS275blctPocO7xpd0a+5fmvPYo+gWeBun7gCUTbKENlqZUcHniHqQEmHI+V+wCp7Kkd/j5ANjq4CAj2SY0CjbKUYFq/anOPuxNwFwB3XA34bwyTIKadiF2gcIzNhIxUVDK9yQWnvC+qJv9YYS6StI0lu9aUQlTC0O0lhQ730prvCUxP53timMZ3qkwDSO96LbDEII9e7Me1rAuxZy+Q9uED5r7J3BjAH8aYYxVKh+n5N2jeED5ef/BM+C5NdFoiv0WdFhmtM6VvzDaui9hHmFWCMLR9XDBlY7YP3dqg014aUgrRCYMXOw741XdcVzeNlcGw4MWwZYKSLJTPSCtdomWpRmyDUhAPEQorQUud/ZqyRjU3ZJ2EQMYm2t5aOGYmZtGJ5qIIqpA9YNMDuD3xS0b6v/ODccpi66AlyBF2AJ1RC17phuHNWuYCbL8wsvVh6qG+8GVvxRJsSMeWyNyd6GpulhcEK368XBw3nBvqe6d9xDt4zF/OPAsWvDQyHlhZIXvITqNJj5LAMA/72FpQBebXfWABvQdMs9sdD7Xz3YPpgWnuSleFY1+X8AS5olnRusTgwHkCy3hvZpmFdUV4pXXAExbYL94o+ifwz4xb9qKrUkfRdYcfjjuFBXLZ/tDr6soSpLst6bPv1KH+Exs/bT3H/I3nIH9ImzAxyS124fcDUI5gQ8f6ZicNVi4u79hCwsoiRemguSGYo17S1JQuMMpU92yZYktl9gzuDFUQoEZUrwLaZLiFDsZKmThzaYx9GVPiPM58BiCTQQJLEdP5/IEFPn3S715eqYsjiskBxaRPsXy3jEOKd8s40EqLQnePwZ5U0RyDruwxFf550T9JPzjQ1NNAbZK8DZ/AJfx+BQtHnb1w2Wavcd979/oKl6CuOZLAGPDA6Gnjo8PaVT+/1dhWF6EzhYLdFTYyp0NIPOluStaellZogIQUTAdp5wHJtZThh+gx3vraAI3TlRmrtXxFFXtqbzDef0OC9lDdcJUF5rWAHUG1u9uMn5WFaHuSF/ZGhKV072KuLMwdHDezHrsoy5xsecjOv7J/H393/jR5/lXTftRqP7u3TNFmOlZR1boz1fj3Xc31Yg5Zh4FvVA96mpt/Q3elSUzZJYL8a6VKNXsLOR00odK88PGwuPH5qPkcsZMRE1WZrnX8aALpKj4V4z+O2E0WT8X47HBwJzuSGyFX6zqeRI8w5udca7eiDeAQJq4xUsbDtNoNR26Qwu/PIFWoq1L7GyVriXs5Yl5LHC1ERRH5Fx7wLa7ZN2aPMSuqy8DKskiVgCl2dPgut9F5xrc/WgyiCt0J2la6IgyNobmCmUbxJBMp38c3boJFj9Uzi+6Ei7JmZF2Lfn+Ll2tfCAa9h75q2x3uLrGBqWvBlQC8aBdH9hu1mvS+kEsybDyNcEIWYts/rNGHlmrfv80Vnetce7KdiutwIz8I7uwH3WhfCbXd2ddX7QVCf+Qzg6Oh7HdHVKPI3TBg9oc/ctAGJ3oJkJh2n4Z0f3E0XISHh9ptBnoQz9sJfnEnmM0VVbmSdAgJPP0WxoDRW7NGy/eQjyb9v4lMwkgwBqoS6nv2e+9Nce/ww8Pb4aHPBQkxNK8DjFrHPaYTaP486IYe+itDMj/g1QccIWDuQZOV4lmAloT2RHR7bUzHAu3Qtag6rkAZPG5eEvfTR5Mz7gHbuaOffbF9PjRFtoskEOg+YF82Tt8HwK9Cm+7+zzuRQRdEmxm7jjJR83QdhBEyDv2bU+YLB+z3f8ixN7CUTsXGipGUmC4ijGQttkgfbz73Mdn28cf30mZLb0og220G7O+sFHVGyyGzqZi9FtVscpq9scdhZun4NaHwiUHhk8UserTE06tzdBv2QbcBcc+HPdCNNugZfTo+QBuTwcYnvw7Cd2Y+Cwz9F4mkg05L1eu/nAc533D16W7/6vqWKx6WiWm39NurhlWKGLuTpzv3SvevfuzEbguLj4vZcVaDIv+XnNbJTMm7spLpZfpeo32kWfG6XName3Rc6dX1NJyjZmB2I88Njt5KNK8GDlLeFtocXfUYPJsK2T2Z45Zk1N527qk9bX55Dn7j4D/FS2hjNiQAAA==',
}
for _name, _b in _FILES.items():
    with open(_name, 'w') as _f:
        _f.write(gzip.decompress(base64.b64decode(_b)).decode('utf-8'))
print('helper modules written:', list(_FILES))

# regenerate the synthetic feature bags if this notebook is run on its own
import pickle
if not os.path.exists('synthetic_bags.pkl'):
    from mil_utils import make_synthetic_dataset
    _d, _ = make_synthetic_dataset(n_patients=80, in_dim=512, seed=1)
    pickle.dump(_d, open('synthetic_bags.pkl', 'wb'))
    print('generated synthetic_bags.pkl')

In [ ]:
import numpy as np, pickle, torch
import matplotlib.pyplot as plt
from mil_models import build_model
from mil_utils import patient_stratified_kfold, train_one, evaluate, compute_metrics

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
with open("synthetic_bags.pkl", "rb") as f:
    data = pickle.load(f)
IN_DIM = data[0]["features"].shape[1]

## 1 · Cross-validated metrics, collected out-of-fold

We gather **out-of-fold** predictions so every slide is scored by a model that didn't see
it, then compute metrics on the pooled predictions and per-fold.

In [ ]:
labels = np.array([d["label"] for d in data])
cw = labels.size / (2.0 * np.maximum(np.bincount(labels, minlength=2), 1))

oof_y, oof_p, per_fold = [], [], []
for tr, va in patient_stratified_kfold(data, n_folds=5, seed=2):
    model = build_model("clam_sb", IN_DIM, 2)
    model, _ = train_one(model, data, tr, va, epochs=25, class_weights=cw,
                         device=DEVICE, patience=8)
    m, y, p = evaluate(model, data, va, device=DEVICE, return_probs=True)
    per_fold.append(m); oof_y += y.tolist(); oof_p += p.tolist()

oof_y, oof_p = np.array(oof_y), np.array(oof_p)
print("=== per-fold (mean ± std) ===")
for key in ["auroc", "auprc", "balanced_acc"]:
    vals = [m[key] for m in per_fold]
    print(f"  {key:12s}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")
print("\n=== pooled out-of-fold ===")
pooled = compute_metrics(oof_y, oof_p)
print({k: round(v, 3) for k, v in pooled.items()})

## 2 · ROC curve and confusion matrix

In [ ]:
def roc_points(y, p):
    thr = np.unique(np.concatenate([[0], p, [1]]))[::-1]
    tpr, fpr = [], []
    P, Nn = y.sum(), (1 - y).sum()
    for t in thr:
        pred = p >= t
        tpr.append((pred & (y == 1)).sum() / max(P, 1))
        fpr.append((pred & (y == 0)).sum() / max(Nn, 1))
    return np.array(fpr), np.array(tpr)

fpr, tpr = roc_points(oof_y, oof_p)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].plot(fpr, tpr, lw=2, label=f"AUROC = {pooled['auroc']:.3f}")
ax[0].plot([0, 1], [0, 1], "--", c="gray"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
ax[0].set_title("ROC (out-of-fold)"); ax[0].legend()

pred = (oof_p >= 0.5).astype(int)
cm = np.array([[((pred == j) & (oof_y == i)).sum() for j in (0, 1)] for i in (0, 1)])
im = ax[1].imshow(cm, cmap="Purples")
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, cm[i, j], ha="center", va="center",
                   color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)
ax[1].set_xticks([0, 1], ["pred benign", "pred tumor"])
ax[1].set_yticks([0, 1], ["true benign", "true tumor"])
ax[1].set_title("Confusion matrix @0.5")
plt.tight_layout(); plt.show()

## 3 · UMAP of patch embeddings

Project pooled patch features to 2-D and colour by tissue type (here: planted-tumor vs
benign). Reveals whether the **encoder** separates tissue — a sanity check independent of
the MIL head. Falls back to PCA if `umap-learn` is unavailable.

In [ ]:
# pool a manageable sample of patches across slides
feats, kind = [], []
rng = np.random.default_rng(0)
for d in rng.choice(data, size=40, replace=False):
    n = min(60, len(d["features"]))
    sel = rng.choice(len(d["features"]), n, replace=False)
    feats.append(d["features"][sel]); kind.append(d["tumor_mask"][sel])
X = np.concatenate(feats); kind = np.concatenate(kind)

try:
    import umap
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X)
    title = "UMAP of patch embeddings"
except Exception:
    Xc = X - X.mean(0); _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    emb = Xc @ Vt[:2].T; title = "PCA of patch embeddings (install umap-learn for UMAP)"

plt.figure(figsize=(6, 5))
plt.scatter(emb[~kind, 0], emb[~kind, 1], s=8, alpha=.4, label="benign")
plt.scatter(emb[kind, 0],  emb[kind, 1],  s=14, c="crimson", label="tumor")
plt.legend(); plt.title(title); plt.tight_layout(); plt.show()

## 4 · The honest-evaluation checklist

- ✅ Split by **patient**, report **mean ± std** across folds.
- ✅ Beat a **mean-pool baseline**.
- ✅ Headline **AUROC**, but also **balanced accuracy / AUPRC** under imbalance.
- ✅ Inspect **attention overlays** (catch shortcut learning) and **UMAP** (encoder quality).
- ✅ Quote an **external-cohort** number (different hospital/scanner) — internal CV is optimistic.

That completes the course pipeline: **data → train → infer → post-process → evaluate.**
Run the **NotebookLM quiz** (`quiz/MIL_quiz_source.md`) to test yourself.